In [25]:
from pybaseball import statcast
from pybaseball import playerid_reverse_lookup
import pandas as pd
from datetime import timedelta


In [13]:
statcast_2025 = statcast(
  start_dt = "2025-03-18",
  end_dt = "2025-09-28"
)

This is a large query, it may take a moment to complete


/Users/JeremyAbbott_1/Library/Python/3.12/lib/python/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
  1%|          | 1/195 [00:01<05:04,  1.57s/it]/Users/JeremyAbbott_1/Library/Python/3.12/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and cat

In [23]:
pa = statcast_2025[statcast_2025["events"].notna()].copy()

pa["game_date"] = pd.to_datetime(pa["game_date"])

pa["week_start"] = (
  pa["game_date"]
  - pd.to_timedelta(pa["game_date"].dt.weekday, unit = "D")
)

weekly = (
  pa.groupby(["batter", "week_start"])
  .agg(
    PA = ("events", "count"),
    H = ("events", lambda x: x.isin(["single", "double", "triple", "home_run"]).sum()),
    BB = ("events", lambda x: x.isin(["walk", "intent_walk"]).sum()),
    HBP = ("events", lambda x: (x == "hit_by_pitch").sum()),
    SF = ("events", lambda x: (x == "sac_fly").sum()),
    AB = ("events", lambda x: (~x.isin(["walk", "intent_walk", "hit_by_pitch", "sac_fly", "sac_bunt", "catcher_interf"])).sum()),
    TB = ("events", lambda x: (
      (x == "single").sum()
      + 2 * (x == "double").sum()
      + 3 * (x == "triple").sum()
      + 4 * (x == "home_run").sum()
    ))
  )
  .reset_index()
)

weekly["AVG"] = weekly["H"] / weekly["AB"]
weekly["OBP"] = (weekly["H"] + weekly["BB"] + weekly["HBP"]) / (
  weekly["AB"] + weekly["BB"] + weekly["HBP"] + weekly["SF"]
)
weekly["SLG"] = weekly["TB"] / weekly["AB"]
weekly["OPS"] = weekly["OBP"] + weekly["SLG"]

weekly = weekly[weekly["PA"] >= 7]

In [28]:
unique_batters = weekly["batter"].unique().tolist()

player_lookup = playerid_reverse_lookup(
  unique_batters,
  key_type = "mlbam"
)

player_lookup["player_name"] = (
  player_lookup["name_first"]
  + " "
  + player_lookup["name_last"]
)

weekly = weekly.merge(
  player_lookup[["key_mlbam", "player_name"]],
  left_on = "batter",
  right_on = "key_mlbam",
  how = "left"
)

weekly = weekly.drop(columns = ["key_mlbam"])

In [32]:
weekly = weekly.sort_values(
  ["batter", "week_start"]
)

weekly["cum_H"] = (
  weekly.groupby("batter")["H"].cumsum()
)

weekly["cum_AB"] = (
  weekly.groupby("batter")["AB"].cumsum()
)

weekly["cum_BB"] = (
  weekly.groupby("batter")["BB"].cumsum()
)

weekly["cum_HBP"] = (
  weekly.groupby("batter")["HBP"].cumsum()
)

weekly["cum_SF"] = (
  weekly.groupby("batter")["SF"].cumsum()
)

weekly["cum_TB"] = (
  weekly.groupby("batter")["TB"].cumsum()
)

weekly["cum_OBP"] = (
  weekly["cum_H"]
  + weekly["cum_BB"]
  + weekly["cum_HBP"]
) / (
  weekly["cum_AB"]
  + weekly["cum_BB"]
  + weekly["cum_HBP"]
  + weekly["cum_SF"]
)

weekly["cum_SLG"] = (
  weekly["cum_TB"]
  / weekly["cum_AB"]
)

weekly["cum_OPS"] = (
  weekly["cum_OBP"]
  + weekly["cum_SLG"]
)

weekly["running_total_OPS"] = (
  weekly.groupby("batter")["cum_OPS"]
  .shift(1)
)

weekly["ops_diff"] = (
  weekly["OPS"]
  - weekly["running_total_OPS"]
)

weekly["ops_pct_diff"] = (
  weekly["OPS"]
  / weekly["running_total_OPS"]
  - 1
)

In [33]:
weekly.to_csv("data/raw/weekly_hitter_data.csv", index=False)

In [35]:
weekly["player_name"] = weekly["player_name"].str.title()
cols_to_round = [
    "AVG",
    "OBP",
    "SLG",
    "OPS",
    "cum_OBP",
    "cum_SLG",
    "cum_OPS",
    "running_total_OPS",
    "ops_diff",
    "ops_pct_diff"
]

weekly[cols_to_round] = weekly[cols_to_round].round(3)

front_cols = [
    "batter",
    "player_name",
    "week_start"
]

remaining_cols = [
    col for col in weekly.columns
    if col not in front_cols
]

weekly = weekly[
    front_cols + remaining_cols
]

weekly = weekly.sort_values(
  ["player_name", "week_start"]
)

In [38]:
analysis_cols = [
  "player_name",
  "week_start",
  "PA",
  "OPS",
  "running_total_OPS",
  "ops_diff",
  "ops_pct_diff"
]

weekly[analysis_cols].to_csv(
  "data/clean/weekly_hitter_data_clean.csv",
  index = False
)